In [2]:
# from gulps.synthesis_plugin import GulpsSynthesisPlugin
import numpy as np
from monodromy.render import _plot_coverage_set
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.circuit.library import (
    CXGate,
    RZXGate,
    UGate,
    UnitaryGate,
    XXPlusYYGate,
    iSwapGate,
)
from qiskit.circuit.random import random_circuit
from qiskit.quantum_info import Operator, average_gate_fidelity
from qiskit.quantum_info.random import random_unitary
from qiskit.transpiler import (
    InstructionProperties,
    PassManager,
    Target,
    generate_preset_pass_manager,
)
from qiskit.transpiler.passes import Optimize1qGatesDecomposition
from tqdm import tqdm
from weylchamber import c1c2c3

from gulps.gulps_synthesis import GulpsDecomposer
from gulps.synthesis_pass import GulpsDecompositionPass
from gulps.utils.invariants import GateInvariants
from gulps.utils.logging_config import logger

# logger.setLevel("DEBUG")
logger.setLevel("INFO")


### Usage as a Decomposer

In [70]:
gate_set = [
    CXGate(),
    # CXGate().power(1 / 2),
    iSwapGate().power(1 / 2),
    # iSwapGate().power(1 / 3),
]
costs = [1.0, 1 / 2]  # , 1 / 2, 1 / 3]
decomposer = GulpsDecomposer(gate_set, costs, precompute_polytopes=True)

In [71]:
N = 2_000
fidelities = []

for idx in tqdm(range(N)):
    u = random_unitary(4, seed=idx)
    v = Operator(decomposer(u))
    fid = average_gate_fidelity(u, v)
    fidelities.append(fid)

    if fid < 1 - 1e-6:
        print(f"Unitary {idx} fidelity is low: {fid:.8f}")
        print("Canonical invariants:")
        print("U:", c1c2c3(u))
        print("V:", c1c2c3(v))
        print("\n")
        continue

# Summary statistics
fidelities = np.array(fidelities)
print(f"\nSummary across {len(fidelities)} samples:")
print(f"  Median fidelity: {np.median(fidelities)}")
print(f"  Mean fidelity:   {np.mean(fidelities)}")
print(f"  Minimum fidelity:{np.min(fidelities)}")

  0%|          | 0/2000 [00:00<?, ?it/s]

[gulps.local_numerics] DEBUG: [EASY 1/4] residual=[ 8.65973959e-15  1.49186219e-15 -7.54951657e-15] (‖residual‖=6.71e-29, nfev=112)
[gulps.local_numerics] DEBUG: => Success on [EASY 1] (componentwise |residual| ≤ 1.0e-08)
[gulps.local_numerics] DEBUG: ✅ LM synthesis SUCCESS on EASY attempt 1 (residual=6.71e-29, total_nfev=112) in 0.008s
[gulps.local_numerics] DEBUG: [EASY 1/4] residual=[-2.10976154e-08  1.04817178e-07 -1.45480885e-08] (‖residual‖=5.82e-15, nfev=256)
[gulps.local_numerics] DEBUG: [EASY 2/4] residual=[-2.97678548e-13  1.61826108e-12 -6.61970478e-14] (‖residual‖=1.36e-24, nfev=92)
[gulps.local_numerics] DEBUG: => Success on [EASY 2] (componentwise |residual| ≤ 1.0e-08)
[gulps.local_numerics] DEBUG: ✅ LM synthesis SUCCESS on EASY attempt 2 (residual=1.36e-24, total_nfev=348) in 0.029s
  0%|          | 1/2000 [00:00<05:45,  5.78it/s][gulps.local_numerics] DEBUG: [EASY 1/4] residual=[ 2.02615702e-14  3.52495810e-15 -1.04360964e-14] (‖residual‖=2.66e-28, nfev=36)
[gulps.local

KeyboardInterrupt: 

In [ ]:
# # %reload_ext  snakeviz
# %load_ext snakeviz
# import cProfile

# u = random_unitary(4, seed=0)
# cProfile.run("decomposer._run(u)", "profile_timings/temph.prof")